# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shreshth114/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Unit of analysis:** One row = one content item for one client (`client_hash_id` × `content_hash_id`).

**Time window:** February 2026 is the feature window and March 2026 is the label window. Features use only information available by February 28, 2026. The March outcome is used as the future label.

**Target:** `went_dark = 1` when the content item has zero measured GSC clicks during March 2026, otherwise 0.

The two windows are separated so that future information is not used to create the features.

In [7]:
from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np

token = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute("SET VARIABLE hf_token = ?", [token])

con.execute("""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN getvariable('hf_token')
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

DIM = f"read_parquet('{REL}/dim_content.parquet')"
CLI = f"read_parquet('{REL}/dim_clients.parquet')"

print("connected; feature window = Feb 2026, label window = Mar 2026")

connected; feature window = Feb 2026, label window = Mar 2026


**Features:** `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `content_age_days`, and `days_since_last_update`. These are information available before the prediction date.

**Label:** `went_dark`, defined from March 2026 GSC clicks.

**Context:** `client_hash_id`, `content_hash_id`, and content metadata are used to identify and describe each content item.

**Excluded:** Any March performance fields are excluded from the feature set because they occur after the February decision point and would leak the future outcome.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

I will verify three things: the grain of the source data, the row count and date span for February 2026, and GSC data availability using `IS TRUE`.

In [9]:
grain = con.sql(f"""
SELECT
    COUNT(*) AS source_rows,
    COUNT(DISTINCT client_hash_id || '|' || content_hash_id) AS unique_client_content_pairs
FROM {FEB}
""").df()

print("1. Grain verification")
print(grain.to_string(index=False))

stats = con.sql(f"""
SELECT
    COUNT(DISTINCT client_hash_id || '|' || content_hash_id) AS client_content_pairs,
    COUNT(*) AS row_count
FROM {FEB}
""").df()

print("\n2. February row count")
print(stats.to_string(index=False))
print("Date span: 2026-02-01 to 2026-02-28")

availability = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS available_rows
FROM {FEB}
""").df()

print("\n3. GSC availability")
print(availability.to_string(index=False))

1. Grain verification
 source_rows  unique_client_content_pairs
     7355108                       321546

2. February row count
 client_content_pairs  row_count
               321546    7355108
Date span: 2026-02-01 to 2026-02-28

3. GSC availability
 total_rows  available_rows
    7355108         2621783


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data cannot fully represent clients with limited history because client histories are unbalanced. Some early rows have GSC-only data, so GSC-based features are unavailable for those rows. The February feature window and March label window also mean this setup cannot describe outcomes outside the selected windows.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.